# 02 · KPI framework and executive metrics

Every number here comes from the **gold tables** through the shared KPI engine
(`metrics/kpi.py` + `metrics/kpis.yml`). That's the same engine the Tableau calculated
fields and the embedded Streamlit app use, so these figures match what executives and
ops managers see.

```
                        PLATFORM MARGIN  (output)
                = trips x avg fare x take rate
         ┌───────────────────┼─────────────────────┐
       TRIPS             AVG FARE              TAKE RATE
  = market x share   = miles x fare/mile   = 1 - driver pay / fare
       │                     │                     │
  service quality      trip mix               driver supply
  avg wait, % <5 min   airport share,         earnings per
  WAV fulfilment       congestion-zone share  engaged hour
```

In [8]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+png"   # interactive in VS Code + a static image GitHub can display

from metrics.kpi import kpi, kpi_names, kpi_sql, spec

con = duckdb.connect(str(ROOT / "data" / "warehouse.duckdb"), read_only=True)
pd.options.display.float_format = "{:,.4f}".format

## 1. The KPI catalog (generated from `metrics/kpis.yml`)

In [9]:
pd.DataFrame([{k: m[k] for k in ("name", "label", "level", "type", "owner", "definition")} for m in spec()["kpis"]])

,name,label,level,type,owner,definition
0,trips,Trips,output,total,"GM, NYC Market",Completed trips. Includes trips flagged by dat...
1,gross_bookings,Gross Bookings,output,total,"Finance, NYC Market","Everything riders paid except tips (fare, toll..."
2,platform_margin,Platform Margin (fare - driver pay),output,total,"Finance, NYC Market","Base fare minus driver pay. Tolls, taxes and f..."
3,market_share,Market Share (trips),output,share_of_total,"GM, NYC Market",The company's share of all Uber + Lyft trips i...
4,avg_fare,Average Base Fare,input,ratio,Pricing,Base passenger fare per trip.
5,fare_per_mile,Fare per Mile,input,ratio,Pricing,Base fare divided by trip miles.
6,avg_trip_miles,Average Trip Miles,input,ratio,Pricing,Miles per trip.
7,take_rate,Take Rate,input,ratio,"Finance, NYC Market",Share of the base fare the platform keeps afte...
8,avg_wait_min,Average Wait (min),input,ratio,Marketplace Ops,Minutes from ride request to pickup. Excludes ...
9,pickup_within_5min_rate,Picked Up Within 5 Min,input,ratio,Marketplace Ops,Share of riders picked up within 5 minutes of ...


## 2. Monthly scorecard by company

In [10]:
names = [n for n in kpi_names() if kpi(n)["type"] != "share_of_total"]
scorecard = con.execute(f"""
    select pickup_month, company,
           {", ".join(f"{kpi_sql(n)} as {n}" for n in names)},
           {kpi_sql('market_share').format(partition='pickup_month')} as market_share
    from gold_monthly_company
    group by pickup_month, company order by pickup_month, company
""").df()
scorecard.set_index(["pickup_month", "company"])[
    ["trips", "market_share", "gross_bookings", "platform_margin", "take_rate", "avg_fare",
     "avg_wait_min", "pickup_within_5min_rate", "driver_earnings_per_engaged_hour"]]

trips  market_share   gross_bookings  \
pickup_month company                                                  
2024-10-01   Lyft     4,882,969.0000        0.2438 154,381,842.1002   
             Uber    15,145,313.0000        0.7562 498,780,700.7696   
2024-11-01   Lyft     4,860,617.0000        0.2432 147,012,930.9402   
             Uber    15,126,916.0000        0.7568 488,322,887.4397   
2024-12-01   Lyft     5,137,337.0000        0.2438 162,572,321.4502   
             Uber    15,931,514.0000        0.7562 541,310,456.2394   
2025-01-01   Lyft     5,049,211.0000        0.2474 142,535,733.0102   
             Uber    15,356,455.0000        0.7526 462,503,532.8501   
2025-02-01   Lyft     4,919,838.0000        0.2544 140,679,155.1402   
             Uber    14,419,623.0000        0.7456 448,777,571.5098   
2025-03-01   Lyft     5,989,698.0000        0.2917 181,775,233.6401   
             Uber    14,547,181.0000        0.7083 504,364,858.2197   

                      platform_margin  take_rate  avg_fare  avg_wait_min  \
pickup_month company                                                       
2024-10-01   Lyft     26,591,832.3200     0.2088   26.0822        5.4915   
             Uber     96,789,895.5000     0.2335   27.4028        4.3421   
2024-11-01   Lyft     28,866,228.1800     0.2362   25.1494        5.4243   
             Uber     86,972,687.9500     0.2143   26.8335        4.4971   
2024-12-01   Lyft     36,271,385.0100     0.2667   26.6340        5.5309   
             Uber     98,803,507.8200     0.2186   28.3836        5.2736   
2025-01-01   Lyft     26,319,757.7200     0.2272   23.0050        4.6915   
             Uber     88,458,564.1200     0.2333   24.6991        4.5554   
2025-02-01   Lyft     24,711,226.6200     0.2152   23.3447        4.8129   
             Uber     88,371,177.4200     0.2402   25.5272        4.7099   
2025-03-01   Lyft     32,345,179.1200     0.2169   25.1560        4.9549   
             Uber    117,439,746.7100     0.2811   28.7259        4.7751   

                      pickup_within_5min_rate  \
pickup_month company                            
2024-10-01   Lyft                      0.5373   
             Uber                      0.7189   
2024-11-01   Lyft                      0.5402   
             Uber                      0.6967   
2024-12-01   Lyft                      0.5363   
             Uber                      0.5935   
2025-01-01   Lyft                      0.6567   
             Uber                      0.6941   
2025-02-01   Lyft                      0.6362   
             Uber                      0.6685   
2025-03-01   Lyft                      0.6160   
             Uber                      0.6634   

                      driver_earnings_per_engaged_hour  
pickup_month company                                    
2024-10-01   Lyft                              62.9676  
             Uber                              64.0260  
2024-11-01   Lyft                              62.5227  
             Uber                              64.8890  
2024-12-01   Lyft                              63.5099  
             Uber                              67.5037  
2025-01-01   Lyft                              63.5124  
             Uber                              65.2458  
2025-02-01   Lyft                              62.4470  
             Uber                              64.7408  
2025-03-01   Lyft                              64.5695  
             Uber                              68.2121

In [11]:
long = scorecard.melt(id_vars=["pickup_month", "company"],
                      value_vars=["market_share", "take_rate", "avg_wait_min", "driver_earnings_per_engaged_hour"])
fig = px.line(long, x="pickup_month", y="value", color="company", facet_col="variable", facet_col_wrap=2,
              markers=True, height=550, title="Headline KPIs by month")
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=kpi(a.text.split("=")[-1])["label"]))
fig.show()

RuntimeError: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"


## 3. What drove the change in platform margin? (Oct 2024 → Mar 2025)

Platform margin = trips × average fare × take rate (on economics-valid trips). Log
differences split the change exactly into the three drivers.

In [ ]:
def decompose(company: str, start: str, end: str) -> pd.DataFrame:
    s = scorecard[scorecard["company"] == company].set_index("pickup_month")
    s.index = s.index.astype(str)
    valid = con.execute(f"""
        select cast(pickup_month as varchar) m, sum(trips_valid_econ) n
        from gold_monthly_company where company = '{company}' group by 1
    """).df().set_index("m")["n"]
    a, b = s.loc[start], s.loc[end]
    parts = {"Trips": np.log(valid[end] / valid[start]),
             "Avg fare": np.log(b["avg_fare"] / a["avg_fare"]),
             "Take rate": np.log(b["take_rate"] / a["take_rate"])}
    total = np.log(b["platform_margin"] / a["platform_margin"])
    change = b["platform_margin"] - a["platform_margin"]
    return pd.DataFrame({"driver": list(parts), "share_of_change": [v / total for v in parts.values()],
                         "usd_impact": [change * v / total for v in parts.values()]}).assign(company=company)

bridge = pd.concat([decompose(c, "2024-10-01", "2025-03-01") for c in ("Uber", "Lyft")])
bridge

,driver,share_of_change,usd_impact,company
0,Trips,-0.2028,"-4,186,839.1505",Uber
1,Avg fare,0.2438,"5,035,303.2777",Uber
2,Take rate,0.9589,"19,801,387.0827",Uber
0,Trips,0.9894,"5,692,113.8910",Lyft
1,Avg fare,-0.1846,"-1,062,117.5458",Lyft
2,Take rate,0.1953,"1,123,350.4547",Lyft


In [ ]:
px.bar(bridge, x="driver", y="usd_impact", color="company", barmode="group",
       title="Platform margin bridge, Oct 2024 → Mar 2025", labels={"usd_impact": "Change in monthly margin ($)", "driver": ""}).show()

## 4. Service quality vs driver earnings, by borough

The same KPI engine works at any grain, here borough × company over the whole window.

In [ ]:
borough = con.execute(f"""
    select z.borough, g.company,
           {kpi_sql('trips')} as trips,
           {kpi_sql('avg_wait_min')} as avg_wait_min,
           {kpi_sql('pickup_within_5min_rate')} as pickup_within_5min_rate,
           {kpi_sql('driver_earnings_per_engaged_hour')} as earnings_per_engaged_hour,
           {kpi_sql('take_rate')} as take_rate
    from gold_daily_zone_company g join dim_zone z on z.zone_id = g.pu_zone_id
    where not z.is_unknown
    group by all order by trips desc
""").df()
px.scatter(borough, x="avg_wait_min", y="earnings_per_engaged_hour", size="trips", color="company", text="borough",
           title="Wait time vs driver earnings by pickup borough").update_traces(textposition="top center").show()
borough

,borough,company,trips,avg_wait_min,pickup_within_5min_rate,earnings_per_engaged_hour,take_rate
0,Manhattan,Uber,"34,170,420.0000",4.4070,0.7053,69.4260,0.3031
1,Brooklyn,Uber,"23,661,067.0000",4.6874,0.6711,59.8057,0.1904
2,Queens,Uber,"19,132,445.0000",5.0195,0.6392,67.8467,0.1945
3,Manhattan,Lyft,"12,326,091.0000",4.9764,0.6125,64.1279,0.2425
4,Bronx,Uber,"12,127,326.0000",4.8505,0.6531,61.3515,0.1354
5,Brooklyn,Lyft,"8,542,453.0000",5.1082,0.5902,59.6090,0.2237
6,Queens,Lyft,"6,330,316.0000",5.2819,0.5762,67.1509,0.2121
7,Bronx,Lyft,"3,230,554.0000",5.4549,0.5333,60.0945,0.2174
8,Staten Island,Uber,"1,432,573.0000",6.0892,0.4934,70.5670,0.1620
9,Staten Island,Lyft,"408,782.0000",6.3367,0.4327,67.3005,0.2380


## 5. Takeaways for the executive dashboard

Fill these in from the outputs above when you present this:

* **Share:** how far Lyft's March share moved against the Oct–Feb run rate.
* **Margin:** which driver (trips, fare or take rate) explains most of each company's margin change.
* **Service:** where waits are longest, and whether those boroughs also pay drivers more per hour.